# TimesFM — Google Pretrained Foundation Model (BONUS, optional)

MLflow experiment: `TimesFM_Training`.
Runs: `TimesFM_ZeroShot`, `TimesFM_Analysis`.

Per `EXPERIMENTS.md` / `TASKS.md` (T17) this is a **bonus** track: Google's pretrained
TimesFM model, used **zero-shot** (no fine-tuning) to forecast the same 39-week horizon.
Metric: WMAE, holiday weeks weighted 5x, same subset-vs-full tradeoff as the other notebooks
(zero-shot forecasting every one of ~3,000 series is slow, so this notebook evaluates on a
representative sample and is explicit about that scope in `TimesFM_Analysis`).

`timesfm` is heavy (torch/jax + a multi-hundred-MB checkpoint download) and is commented out
in `requirements-dl.txt` by default - uncomment it there before running this notebook.

## Environment

In [ ]:
!pip -q uninstall -y timesfm
!pip -q install git+https://github.com/google-research/timesfm.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import timesfm

print(timesfm.__file__)
print([x for x in dir(timesfm) if "Times" in x or "Hparams" in x or "Checkpoint" in x])

In [ ]:
import timesfm
import inspect

print("timesfm file:", timesfm.__file__)
print("available TimesFM objects:")
print([x for x in dir(timesfm) if "Times" in x or "times" in x or "Hparams" in x or "Checkpoint" in x])

cls = timesfm.TimesFM_2p5_200M_torch
print("class:", cls)
print("init signature:")
print(inspect.signature(cls))
print("methods:")
print([m for m in dir(cls) if not m.startswith("_")])

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install mlflow dagshub kaggle pandas pyarrow python-dotenv timesfm
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("..")
print("IN_COLAB =", IN_COLAB)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
print("IN_COLAB =", IN_COLAB, "| ROOT =", ROOT)


## Robust data directory (Fix 1)

In [ ]:
from pathlib import Path

def find_data_dir(root):
    """Locate the folder that actually holds the 5 Kaggle CSVs.

    Handles both layouts: a flat data/ folder, or the nested folder Kaggle's
    zip produces (data/walmart-recruiting-store-sales-forecasting/).
    """
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("DATA_DIR =", DATA_DIR)


## MLflow / DagsHub tracking (Fix 2)

In [ ]:
!pip -q install dagshub mlflow

import os
import mlflow
import dagshub
from google.colab import userdata

# Use your exact Colab secret names here.
# If your secrets are currently named DAGSHUE..., either rename them or change these two lines.
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

dagshub.init(
    repo_owner="karev23",
    repo_name="ML-final",
    mlflow=True,
)

# Force MLflow away from local ./mlruns
mlflow.set_tracking_uri("https://dagshub.com/karev23/ML-final.mlflow")

print("Tracking URI:", mlflow.get_tracking_uri())

mlflow.set_experiment("TimesFM_Training")

In [ ]:
import os
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on already-exported env vars")

# stale proxy vars have previously broken the DagsHub connection in Colab
for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

EXPERIMENT_NAME = "TimesFM_Training"

def setup_tracking():
    """Use DagsHub MLflow if creds are present in the env / .env, else fall
    back to a local file-based tracking store so a bad token or a network
    hiccup never blocks a training run."""
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if uri and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
        try:
            mlflow.set_tracking_uri(uri)
            mlflow.set_experiment(EXPERIMENT_NAME)
            print("MLflow tracking ->", uri)
            return
        except Exception as e:
            print("DagsHub tracking unavailable, falling back to local mlruns:", e)
    fallback_uri = f"file:{(ROOT / 'mlruns').as_posix()}"
    mlflow.set_tracking_uri(fallback_uri)
    mlflow.set_experiment(EXPERIMENT_NAME)
    print("MLflow tracking -> local", fallback_uri)

setup_tracking()

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)


## Imports and data

`timesfm` import is optional-but-required: this cell fails loudly with install instructions
rather than silently no-op'ing, since a zero-shot notebook with no model is pointless - but it
does not touch the DL-only requirements file if you only want to read the analysis below.

In [ ]:

import os
import shutil
import glob
import zipfile
from pathlib import Path
from google.colab import userdata

# Reset to the real repo root
ROOT = Path("/content/ML-final")
os.chdir(ROOT)

DATA_DIR = ROOT / "data"

# Remove broken nested data folder and recreate clean data/
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle credentials
# Use exact secret names. If your secret is KAGGLE_JSON, use the JSON version below.
import json

kaggle_json = userdata.get("KAGGLE_JSON")
if kaggle_json:
    kaggle = json.loads(kaggle_json)
    os.environ["KAGGLE_USERNAME"] = kaggle["username"]
    os.environ["KAGGLE_KEY"] = kaggle["key"]
else:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("cwd:", os.getcwd())
print("data dir:", DATA_DIR)
print("kaggle user:", os.environ["KAGGLE_USERNAME"])

# Download and unzip
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p /content/ML-final/data

for zip_path in glob.glob("/content/ML-final/data/*.zip"):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content/ML-final/data")

for zip_path in glob.glob("/content/ML-final/data/*.csv.zip"):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content/ML-final/data")

print("Final data folder:")
!find /content/ML-final/data -maxdepth 2 -type f -name "*.csv" -print

In [ ]:
DATA_DIR = Path("/content/ML-final/data")

train, test, features, stores, sample = load_raw(DATA_DIR)

print("train:", train.shape)
print("test:", test.shape)
print("features:", features.shape)
print("stores:", stores.shape)
print("sample:", sample.shape)

## Subset selection

Same rule as the Prophet notebook: zero-shot forecasting every series is unnecessary to make
the point of this bonus track, so evaluate on the `N_SUBSET` highest-volume series.

In [ ]:
N_SUBSET = 4
series_totals = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"].sum().sort_values(ascending=False)
)
subset_keys = series_totals.head(N_SUBSET).index.tolist()
print("selected (Store, Dept) series:", subset_keys)

## Run: TimesFM_ZeroShot

For each subset series: hold out the last `HORIZON` weeks, feed everything before that as
**context** to the pretrained checkpoint, and forecast forward with no fine-tuning and no
exogenous features (TimesFM's public zero-shot API is univariate).

In [ ]:
def zero_shot_forecast(tfm_model, context_values, horizon):
    context_values = np.asarray(context_values, dtype=float)

    forecast = tfm_model.forecast(
        inputs=[context_values],
        freq=[0],
    )

    if isinstance(forecast, tuple):
        forecast = forecast[0]

    return np.clip(np.asarray(forecast[0][:horizon], dtype=float), 0, None)


zero_shot_scores = []

if TIMESFM_AVAILABLE:
    tfm = timesfm.TimesFM_2p5_200M_torch(
        torch_compile=False,
        config={
            "horizon_len": HORIZON,
            "context_len": 512,
        },
    )

    with mlflow.start_run(run_name="TimesFM_ZeroShot"):
        safe_log_param("checkpoint", "google/timesfm-2.5-200m")
        safe_log_param("n_subset", N_SUBSET)
        safe_log_param("fine_tuned", False)
        safe_log_param("uses_exogenous", False)

        for i, (store, dept) in enumerate(subset_keys):
            s = train[
                (train["Store"] == store) & (train["Dept"] == dept)
            ].sort_values("Date")

            tr_mask, va_mask = holdout_split(
                s["Date"].to_numpy(),
                horizon=HORIZON,
            )

            tr, va = s[tr_mask], s[va_mask]

            if len(tr) < HORIZON or len(va) == 0:
                continue

            p = zero_shot_forecast(
                tfm,
                tr["Weekly_Sales"].to_numpy(dtype=float),
                len(va),
            )

            hva = va["IsHoliday"].astype(bool).to_numpy()
            score = wmae(va["Weekly_Sales"].to_numpy(), p, hva)

            zero_shot_scores.append(score)
            safe_log_metric(f"wmae_series_{i}", score)

            print(f"Store {store} Dept {dept}: WMAE = {score:,.1f}")

        mean_zero_shot = (
            float(np.mean(zero_shot_scores))
            if zero_shot_scores
            else float("nan")
        )

        safe_log_metric("wmae_subset_mean", mean_zero_shot)
        print("subset mean zero-shot WMAE:", round(mean_zero_shot, 1))

else:
    mean_zero_shot = float("nan")
    print("Skipped: timesfm not installed. See the import cell above.")


## Run: TimesFM_Analysis

Written comparison: log the subset WMAE alongside a short analysis of *why* a general
pretrained model can be expected to underperform a model tuned on this exact dataset, as an
artifact for the README.

In [ ]:
ANALYSIS_TEMPLATE = """TimesFM zero-shot analysis
==========================

Subset mean WMAE (zero-shot, no fine-tuning): {wmae:.1f}

Why a general pretrained model can underperform a model tuned on this exact data:

1. No exogenous signal. TimesFM's public zero-shot interface forecasts a univariate
   series from its own history. It never sees Temperature, Fuel_Price, CPI,
   Unemployment, or the MarkDown promo columns that features.py builds for the tree
   models - all of which measurably move Walmart weekly sales.
2. No holiday awareness. It has no notion that a given week is Super Bowl,
   Thanksgiving, or Christmas week, so it cannot learn the outsized holiday-week sales
   spikes the way build_features' explicit holiday flags let the tree models do -
   and WMAE weights exactly those weeks 5x.
3. No dataset-specific tuning. It was pretrained on a broad, generic corpus of time
   series, not on Walmart retail sales, so it cannot exploit this dataset's specific
   structure (year-over-year seasonality, per-store scale, per-department behavior)
   the way a lag-52 feature or a fitted LightGBM/XGBoost model does.
4. No cross-series learning tailored to this panel. Even though TimesFM is a global
   foundation model, it is not adapted to *this* panel's store/department structure
   the way a model trained end-to-end on it (LightGBM, XGBoost, or even DLinear/
   N-BEATS/PatchTST trained on this data) is.

Where it could still help: as a zero-cost baseline forecast (no training time at all),
as a sanity check against the seasonal-naive baseline, or as one more input feature to a
meta-model / ensemble on top of the tuned tree and DL models.
"""

ANALYSIS_TEXT = ANALYSIS_TEMPLATE.format(wmae=mean_zero_shot)

with open("timesfm_analysis.txt", "w") as f:
    f.write(ANALYSIS_TEXT)

with mlflow.start_run(run_name="TimesFM_Analysis"):
    safe_log_metric("wmae_subset_mean", mean_zero_shot)
    safe_log_artifact("timesfm_analysis.txt")

print(ANALYSIS_TEXT)

## Results

- Subset zero-shot WMAE: ___  | seasonal-naive baseline WMAE on the same subset: ___
- Did TimesFM beat the naive baseline: ___
- Did TimesFM beat the tuned tree models on the same subset weeks: ___ (expected: no)
- Takeaway for the README's bonus section: ___
